In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import pymongo

## 매장 정보를 얻어오기 위해서 시도코드와 시도코드에 따른 구군코드를 얻어오기 위해 MongoDB에 저장된 데이터를 얻어온다.

In [5]:
conn = pymongo.MongoClient('mongodb://localhost:27017') # MongoDB와 연결한다.
starbucks = conn['starbucks'] # MongoDB에 연결된 상태에서 starbucks 데이터베이스와 연결한다.
gugunAll = starbucks['gugunAll'] # starbucks 데이터베이스와 연결된 상태에서 gugunAll 컬렉션에 연결한다.
gugunList = gugunAll.find({}, {'_id': 0}) # starbucks> db.gugunAll.find({}, {'_id': 0})
print(gugunList)

# for gugun in gugunList:
    # print(gugun)

In [6]:
sidoSet = set()
for gugun in gugunList:
    sidoSet.add(gugun['sido_nm'])
print(sidoSet)

{'경기', '경북대구통합특별시', '전남광주', '경남', '전북', '충남', '충북', '세종', '인천', '강원', '대전', '울산', '부산', '제주', '서울'}


## 시도 이름과 구군 이름을 입력받아 매장 정보를 얻어와서 MongoDB에 저장할 도큐먼트를 만들어 저장한다.

시도 이름을 보여주고 매장 정보를 검색할 시도 이름을 입력받는다.

In [58]:
print('시도 이름')
for sido in sidoSet:
    print(f'[{sido}]', end=' ')
print()
sido_nm = input('매장 정보를 검색하려는 시도 이름을 입력하세요: ')

시도 이름
[경기] [경북대구통합특별시] [전남광주] [경남] [전북] [충남] [충북] [세종] [인천] [강원] [대전] [울산] [부산] [제주] [서울] 
매장 정보를 검색하려는 시도 이름을 입력하세요: 경기


입력받은 시도 이름에 따른 구군 이름을 보여주고 검색할 구군 정보를 얻어온다.

In [59]:
gugunList = gugunAll.find({'sido_nm': sido_nm}, {'_id': 0}) # starbucks> db.gugunAll.find({'sido_nm': '서울'}, {'_id': 0})
print('구군 이름')
for gugun in gugunList:
    # print(gugun)
    print(f'[{gugun["gugun_nm"]}]', end=' ')
print()
gugun_nm = input('매장 정보를 검색하려는 구군 이름을 입력하세요: ')

구군 이름
[가평군] [고양시 덕양구] [고양시 일산동구] [고양시 일산서구] [과천시] [광명시] [광주시] [구리시] [군포시] [김포시] [남양주시] [동두천시] [부천시] [성남시 분당구] [성남시 수정구] [성남시 중원구] [수원시 권선구] [수원시 영통구] [수원시 장안구] [수원시 팔달구] [시흥시] [안산시 단원구] [안산시 상록구] [안성시] [안양시 동안구] [안양시 만안구] [양주시] [양평군] [여주시] [연천군] [오산시] [용인시 기흥구] [용인시 수지구] [용인시 처인구] [의왕시] [의정부시] [이천시] [파주시] [평택시] [포천시] [하남시] [화성시] 
매장 정보를 검색하려는 구군 이름을 입력하세요: 수원


구군 코드를 사용하면 아래의 코드처럼 간단하게 검색할 수 있지만 우리가 입력받은 정보는 `시도 이름`과 `구군 이름`밖에 없기 때문에 사용할 수 없다.
`gugunAll.find({'gugun_cd': gugun['gugun_cd']}, {'_id': 0}).next() # starbucks> db.gugunAll.find({'gugun_cd': '0101'}, {'_id': 0})`

사용하는 조건이 `eq`와 `and` 조건일 경우 아래의 코드처럼 검색할 수 있다.  
`gugunAll.find({'sido_nm': sido_nm, 'gugun_nm': gugun_nm}, {'_id': 0}) # db.gugunAll.find({'sido_nm': '서울', 'gugun_nm': '강남구'}, {'_id': 0})`

사용하는 조건이 `eq`와 `and`만 사용하지 않기 때문에 아래의 코드도 사용할 수 있어야 한다.
```
db.gugunAll.find(
  {
  	$and: [
    	{'sido_nm': {$eq: '서울'}},
    	{'gugun_nm': {$eq: '강남구'}}
  	]
  }, 
  {'_id': 0}
)
```

경기도의 경우 고양시, 성남시, 수원시, 안산시는 시를 구성하는 구가 존재하므로 고양시, 성남시, 수원시, 안산시만 입력하면 입력한 내용으로 `시작`하는 모든 구군 정보를 `정규식`을 사용해서 얻어올 수 있다.

```
db.gugunAll.find(
  {
  	$and: [
    	{'sido_nm': {$eq: '경기'}},
    	{'gugun_nm': {$regex: '^성남시'}}
  	]
  }, 
  {'_id': 0}
)
```

In [61]:
gugun = gugunAll.find(
  {
      '$and': [
        {'sido_nm': {'$eq': sido_nm}},
        {'gugun_nm': {'$regex': '^' + gugun_nm}}
      ]
  }, 
  {'_id': 0}
)

# for gu in gugun:
    # print(gu)

시도 코드와 구군 코드를 이용해서 매장 정보를 크롤링한다.